In [1]:
# Step 0 (CPU only): can we build a scanner-shift benchmark from the RSNA knee training set?
# Inputs attached in the Kaggle notebook:
#   1. competition data: rsna-knee-abnormality-detection
#   2. dataset: tascj0/rsna-2026-knee-agent-labels  (Astra labels, CC BY 4.0)
# Reads ONE DICOM header per study (no pixels), in parallel. Outputs:
#   A. which site / scanner fields survive anonymisation
#   B. scanner groups and per-finding positive counts per group
#   C. a candidate held-out split with enough positives on both sides
#   D. metadata-only baseline: AUC of predicting each finding from scanner fields alone
import os, glob, time
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import pydicom
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
TRAIN = os.path.join(COMP, "train_series")
_lab = glob.glob("/kaggle/input/**/labels_astra_trained_guide.csv", recursive=True)
assert _lab, "labels_astra_trained_guide.csv not found: attach tascj0/rsna-2026-knee-agent-labels"
LABELS = _lab[0]
print("labels file:", LABELS, flush=True)

TAGS = ["InstitutionName", "InstitutionAddress", "StationName", "Manufacturer",
        "ManufacturerModelName", "MagneticFieldStrength", "SoftwareVersions",
        "ImagingFrequency", "ReceiveCoilName", "DeviceSerialNumber"]
FP_TAGS = ["Manufacturer", "ManufacturerModelName", "MagneticFieldStrength",
           "SoftwareVersions", "ReceiveCoilName"]
MIN_POS = 20      # minimum positives per finding on the held-out side
N_THREADS = 16

# ---------- A. read one header per study ----------
series = pd.read_csv(os.path.join(COMP, "train_series.csv"))
first_series = series.groupby("StudyInstanceUID")["SeriesInstanceUID"].first()

def first_file(path):
    """Return the first DICOM file under path, descending at most two levels."""
    for _ in range(3):
        with os.scandir(path) as it:
            entries = list(it)
        files = [e.path for e in entries if e.is_file() and e.name.lower().endswith(".dcm")]
        if files:
            return files[0]
        dirs = [e.path for e in entries if e.is_dir()]
        if not dirs:
            return None
        path = dirs[0]
    return None

def read_study(item):
    study, ser = item
    path = os.path.join(TRAIN, study, ser)
    if not os.path.isdir(path):
        path = os.path.join(TRAIN, study)
    try:
        f = first_file(path)
        if f is None:
            return None
        ds = pydicom.dcmread(f, stop_before_pixels=True)
    except Exception as e:
        return {"StudyInstanceUID": study, "error": repr(e)}
    row = {"StudyInstanceUID": study}
    for t in TAGS:
        v = getattr(ds, t, None)
        row[t] = str(v).strip() if v not in (None, "") else None
    return row

items = list(first_series.items())
t0 = time.time()
test = [read_study(it) for it in items[:20]]
dt = (time.time() - t0) / 20
print(f"trial: {dt:.2f} s/study sequential; estimated {dt * len(items) / N_THREADS / 60:.1f} min with {N_THREADS} threads", flush=True)
print(pd.DataFrame([r for r in test if r]).head(3).T, flush=True)

rows, t0 = [], time.time()
with ThreadPoolExecutor(N_THREADS) as ex:
    for i, r in enumerate(ex.map(read_study, items)):
        if r:
            rows.append(r)
        if (i + 1) % 200 == 0:
            print(f"  read {i + 1}/{len(items)}  ({time.time() - t0:.0f} s)", flush=True)
meta = pd.DataFrame(rows)
if "error" in meta:
    print(f"errors: {meta['error'].notna().sum()}  e.g. {meta['error'].dropna().head(2).tolist()}")
meta.to_csv("/kaggle/working/study_site_metadata.csv", index=False)
print(f"\n{len(meta)} studies read")
print("\n== A. field coverage")
for t in TAGS:
    print(f"{t:28s} filled {meta[t].notna().mean():5.0%}   distinct {meta[t].nunique()}")

# ---------- join Astra labels ----------
lab = pd.read_csv(LABELS)
id_col = next(c for c in lab.columns if "study" in c.lower())
findings = [c for c in lab.columns if c != id_col and pd.api.types.is_numeric_dtype(lab[c])]
print(f"\nlabel id column: {id_col}; findings: {findings}")
lab[id_col] = lab[id_col].astype(str)
df = meta.merge(lab, left_on="StudyInstanceUID", right_on=id_col, how="inner")
print(f"studies with labels: {len(df)}")
hard = (df[findings] >= 0.5).astype(int)  # for counting and the metadata baseline only

# ---------- B. groups and positives per group ----------
for name, cols in [("site", ["InstitutionName"]), ("scanner", FP_TAGS[:3]), ("fingerprint", FP_TAGS)]:
    if df[cols[0]].notna().mean() < 0.5:
        print(f"\n== B. {name}: skipped (field mostly empty)")
        continue
    df[name] = df[cols].astype(str).agg(" | ".join, axis=1)
    g = hard.groupby(df[name]).sum()
    g.insert(0, "n_studies", df.groupby(name).size())
    g = g.sort_values("n_studies", ascending=False)
    g.to_csv(f"/kaggle/working/positives_by_{name}.csv")
    print(f"\n== B. {name}: {len(g)} groups, {(g.n_studies >= 200).sum()} with >= 200 studies")
    with pd.option_context("display.width", 250, "display.max_columns", 20):
        print(g.head(15).to_string())

# ---------- C. candidate held-out split (by the coarsest usable grouping) ----------
group_col = next((c for c in ["site", "scanner", "fingerprint"] if c in df.columns), None)
if group_col:
    order = df[group_col].value_counts()
    held, n = [], 0
    for grp, cnt in order.iloc[::-1].items():  # add smaller groups first, aim for ~15-25% of studies
        if n + cnt > 0.25 * len(df):
            continue
        held.append(grp)
        n += cnt
        if n >= 0.15 * len(df):
            break
    mask = df[group_col].isin(held)
    pos_held, pos_train = hard[mask].sum(), hard[~mask].sum()
    print(f"\n== C. held-out {group_col} groups: {len(held)}, studies {mask.sum()} ({mask.mean():.0%})")
    print(pd.DataFrame({"pos_heldout": pos_held, "pos_train": pos_train,
                        "ok": pos_held >= MIN_POS}).to_string())
    pd.Series(held, name=group_col).to_csv("/kaggle/working/heldout_groups.csv", index=False)
else:
    print("\n== C. no usable grouping found")

# ---------- D. metadata-only baseline (no pixels) ----------
X = pd.get_dummies(df[FP_TAGS].fillna("NA").astype(str))
print(f"\n== D. metadata-only AUC (scanner fields -> finding), 5-fold CV, {X.shape[1]} one-hot features")
res = {}
for f in findings:
    y = hard[f].values
    if y.sum() < 10 or y.sum() > len(y) - 10:
        continue
    p = cross_val_predict(LogisticRegression(max_iter=2000, C=0.5), X, y,
                          cv=StratifiedKFold(5, shuffle=True, random_state=0),
                          method="predict_proba")[:, 1]
    res[f] = roc_auc_score(y, p)
res = pd.Series(res).sort_values(ascending=False)
print(res.round(3).to_string())
print(f"macro: {res.mean():.3f}  (0.5 = scanner fields carry no label information)")
res.to_csv("/kaggle/working/metadata_only_auc.csv")

labels file: /kaggle/input/datasets/tascj0/rsna-2026-knee-agent-labels/labels_astra_trained_guide.csv
trial: 0.01 s/study sequential; estimated 0.1 min with 16 threads
                                                                       0  \
StudyInstanceUID       1.2.826.0.1.3680043.8.498.10004873229099053869...   
InstitutionName                                                     None   
InstitutionAddress                                                  None   
StationName                                                         None   
Manufacturer                                          Philips Healthcare   
ManufacturerModelName                                            Ingenia   
MagneticFieldStrength                                                1.5   
SoftwareVersions                                    ['5.6.1', '5.6.1.3']   
ImagingFrequency                                                63.87066   
ReceiveCoilName                                         SENSE_KNEE_16_AC

In [2]:
# Stage 0 data check for FAAC (CPU only, no pixels).
# Needs: competition data, tascj0/rsna-2026-knee-agent-labels, and /kaggle/working/scanner_split.csv
# (re-run the split cell first if the session restarted).
# Answers:
#   1. series composition per vendor (planes x fluid / fat-suppression)
#   2. FAAC v0 rules: how many comparable series pairs per study and finding, per vendor, vs uniform pairs
#   3. ROI-box files: schema, and box coverage per vendor (is the detector worse on held-out vendors?)
#   4. which vendors the agent-drawn boxes (detector training data) came from
#   5. does the vendor stay the same across all series of a study (sample check)
import os, glob, itertools
import numpy as np
import pandas as pd
import pydicom

from concurrent.futures import ThreadPoolExecutor

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
TRAIN = os.path.join(COMP, "train_series")
LABDIR = os.path.dirname(glob.glob("/kaggle/input/**/labels_astra_trained_guide.csv", recursive=True)[0])
series = pd.read_csv(os.path.join(COMP, "train_series.csv"))

# ---------- 0. vendor per study and the scanner split (self-contained) ----------
META = "/kaggle/working/study_site_metadata.csv"
if os.path.exists(META):
    meta = pd.read_csv(META)
else:
    def read_one(item):
        study, ser = item
        d = os.path.join(TRAIN, study, ser)
        with os.scandir(d) as it:
            f = next((e.path for e in it if e.is_file() and e.name.lower().endswith(".dcm")), None)
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        return {"StudyInstanceUID": study,
                **{t: str(getattr(ds, t, "")).strip() or None
                   for t in ["Manufacturer", "ManufacturerModelName", "MagneticFieldStrength"]}}
    items = list(series.groupby("StudyInstanceUID")["SeriesInstanceUID"].first().items())
    with ThreadPoolExecutor(16) as ex:
        meta = pd.DataFrame(list(ex.map(read_one, items)))
    meta.to_csv(META, index=False)

def vendor(m):
    m = str(m).lower()
    if "siemens" in m: return "Siemens"
    if "philips" in m: return "Philips"
    if m.startswith("ge") or "ge medical" in m: return "GE"
    if "toshiba" in m or "canon" in m: return "Canon/Toshiba"
    if "hitachi" in m or "fujifilm" in m: return "Hitachi/Fujifilm"
    return "Other"
meta["vendor"] = meta["Manufacturer"].map(vendor)
rng0 = np.random.RandomState(0)
meta["split"] = meta["vendor"].map({"GE": "test_GE", "Canon/Toshiba": "test_Canon"}).fillna("test_other")
in_dist = meta.index[meta["vendor"].isin(["Siemens", "Philips"])]
meta.loc[in_dist, "split"] = "train"
meta.loc[rng0.choice(in_dist, int(0.15 * len(in_dist)), replace=False), "split"] = "val_id"
split = meta[["StudyInstanceUID", "Manufacturer", "ManufacturerModelName", "MagneticFieldStrength", "vendor", "split"]]
split.to_csv("/kaggle/working/scanner_split.csv", index=False)
print("== 0. split sizes"); print(split.groupby(["split", "vendor"]).size().to_string(), "\n")
s = series.merge(split[["StudyInstanceUID", "vendor", "split"]], on="StudyInstanceUID", how="left")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

# ---------- 1. series composition per vendor ----------
s["kind"] = (s["Anatomical_Plane"].str[:3] + "_"
             + np.where(s["Fluid_Sensitive"] == 1, "fluid", "struct")
             + np.where(s["Fat_Suppression"] == 1, "FS", ""))
print("== 1. series per study by vendor")
print(s.groupby("vendor").agg(studies=("StudyInstanceUID", "nunique"),
                              series=("SeriesInstanceUID", "size")).assign(
      per_study=lambda d: (d.series / d.studies).round(1)).to_string())
comp = (s.groupby(["vendor", "StudyInstanceUID"])["kind"].apply(set)
         .reset_index())
kinds = sorted(s["kind"].unique())
for k in kinds:
    comp[k] = comp["kind"].apply(lambda x: k in x)
print("\nfraction of studies having each series kind, by vendor")
print(comp.groupby("vendor")[kinds].mean().round(2).to_string())

# ---------- 2. FAAC v0 comparable pairs ----------
FLUID = ["Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = {"ACL": ["Sagittal"], "MCL": ["Coronal"],
          "Medial Meniscus": ["Sagittal", "Coronal"], "Lateral Meniscus": ["Sagittal", "Coronal"],
          "Medial OA": ["Coronal", "Sagittal"], "Lateral OA": ["Coronal", "Sagittal"],
          "PF OA": ["Axial", "Sagittal"]}

def faac_pair(a, b, finding):
    if finding in FLUID:
        return a.Fluid_Sensitive == 1 and b.Fluid_Sensitive == 1 and a.Fat_Suppression == 1 and b.Fat_Suppression == 1
    return a.Anatomical_Plane == b.Anatomical_Plane and a.Anatomical_Plane in PLANES[finding]

findings = FLUID + list(PLANES)
rows = []
for study, g in s.groupby("StudyInstanceUID"):
    recs = list(g.itertuples())
    pairs = list(itertools.combinations(recs, 2))
    row = {"StudyInstanceUID": study, "vendor": recs[0].vendor, "uniform": len(pairs)}
    for f in findings:
        row[f] = sum(faac_pair(a, b, f) for a, b in pairs)
    rows.append(row)
pairs = pd.DataFrame(rows)
pairs.to_csv("/kaggle/working/faac_pair_counts.csv", index=False)
print("\n== 2. mean comparable pairs per study (uniform = all pairs)")
print(pairs.groupby("vendor")[["uniform"] + findings].mean().round(2).to_string())
print("\nfraction of studies with >= 1 comparable pair")
print((pairs[["uniform"] + findings] > 0).groupby(pairs["vendor"]).mean().round(2).to_string())

# ---------- 3. ROI-box files: schema and coverage per vendor ----------
for name in ["roi_boxes_detector.parquet", "roi_boxes_agent.parquet", "knee_side.csv"]:
    p = os.path.join(LABDIR, name)
    d = pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p)
    print(f"\n== 3. {name}: {d.shape}\ncolumns: {list(d.columns)}")
    print(d.head(3).to_string())

det = pd.read_parquet(os.path.join(LABDIR, "roi_boxes_detector.parquet"))
ser_col = next(c for c in det.columns if "series" in c.lower())
det = det.merge(s[["SeriesInstanceUID", "vendor", "kind"]], left_on=ser_col, right_on="SeriesInstanceUID", how="left")
num = [c for c in det.columns if pd.api.types.is_float_dtype(det[c])]
print(f"\nnumeric box columns: {num}")
print("detector rows per series and numeric column means, by vendor")
print(det.groupby("vendor")[num].mean().round(3).to_string())
print(det.groupby("vendor")[ser_col].nunique().rename("series_with_boxes").to_string())

# ---------- 4. which vendors the agent boxes came from ----------
agent = pd.read_parquet(os.path.join(LABDIR, "roi_boxes_agent.parquet"))
a_ser = next(c for c in agent.columns if "series" in c.lower())
agent_v = agent[[a_ser]].drop_duplicates().merge(s[["SeriesInstanceUID", "vendor"]],
                                                 left_on=a_ser, right_on="SeriesInstanceUID", how="left")
print("\n== 4. agent-drawn box series by vendor (detector training data)")
print(agent_v["vendor"].value_counts(dropna=False).to_string())

# ---------- 5. vendor consistency within a study (sample of 200 studies) ----------
TRAIN = os.path.join(COMP, "train_series")
def first_dcm(path):
    with os.scandir(path) as it:
        for e in it:
            if e.is_file() and e.name.lower().endswith(".dcm"):
                return e.path
    return None
rng = np.random.RandomState(0)
mismatch, checked = 0, 0
for study in rng.choice(s["StudyInstanceUID"].unique(), 200, replace=False):
    mans = set()
    for ser in s.loc[s.StudyInstanceUID == study, "SeriesInstanceUID"]:
        p = os.path.join(TRAIN, study, ser)
        f = first_dcm(p) if os.path.isdir(p) else None
        if f:
            mans.add(str(getattr(pydicom.dcmread(f, stop_before_pixels=True), "Manufacturer", "")).lower()[:5])
    checked += 1
    mismatch += len(mans) > 1
print(f"\n== 5. studies with >1 manufacturer across series: {mismatch}/{checked}")

== 0. split sizes
split       vendor          
test_Canon  Canon/Toshiba        226
test_GE     GE                   905
test_other  Hitachi/Fujifilm      24
train       Philips             1127
            Siemens             1638
val_id      Philips              174
            Siemens              313 

== 1. series per study by vendor
                  studies  series  per_study
vendor                                      
Canon/Toshiba         226    1343        5.9
GE                    905    5101        5.6
Hitachi/Fujifilm       24     130        5.4
Philips              1301    7607        5.8
Siemens              1951   10190        5.2

fraction of studies having each series kind, by vendor
                  Axi_fluidFS  Axi_struct  Cor_fluidFS  Cor_struct  Sag_fluidFS  Sag_struct
vendor                                                                                     
Canon/Toshiba             1.0        0.41         1.00        0.42         1.00        1.00
GE          

In [3]:
# FAAC pilot, step 1: frozen-feature extraction + timing.
# Kaggle notebook settings: Accelerator = GPU T4 x1, Internet = ON (to download ImageNet weights).
# Inputs: competition data (rsna-knee-abnormality-detection) + tascj0/rsna-2026-knee-agent-labels.
#
# For every series: read all slices, sort by physical position, subsample to <= MAX_SLICES,
# window at the series' 0.5/99.5 percentiles, resize to 224, run a frozen ImageNet ConvNeXt-T,
# then pool the last feature map (a) globally and (b) inside each finding's detector ROI box.
# Saves one .npz per study: per series a [13, 768] float16 array (12 box features + 1 global).
#
# N_STUDIES = 20 is the timing run. Set N_STUDIES = None later for the full extraction.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)

import os, glob, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F
import timm

N_STUDIES = None
MAX_SLICES = 32
IMG = 224
OUT = "/kaggle/working/feats"
os.makedirs(OUT, exist_ok=True)

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
TRAIN = os.path.join(COMP, "train_series")
LABDIR = os.path.dirname(glob.glob("/kaggle/input/**/labels_astra_trained_guide.csv", recursive=True)[0])
FINDINGS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
            "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

series = pd.read_csv(os.path.join(COMP, "train_series.csv"))
boxes = pd.read_parquet(os.path.join(LABDIR, "roi_boxes_detector.parquet"))
box_lookup = {sid: g.set_index("label")[["x0", "y0", "z0", "x1", "y1", "z1"]]
              for sid, g in boxes.groupby("SeriesInstanceUID")}

# ---------- model ----------
dev = "cuda"
model = timm.create_model("convnext_tiny.in12k_ft_in1k", pretrained=True, num_classes=0).to(dev).eval().half()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1).half()
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1).half()

# ---------- reading ----------
def slice_position(ds):
    try:
        o = np.array(ds.ImageOrientationPatient, dtype=float)
        normal = np.cross(o[:3], o[3:])
        return float(np.dot(np.array(ds.ImagePositionPatient, dtype=float), normal))
    except Exception:
        return float(getattr(ds, "InstanceNumber", 0))

def read_series(study, ser):
    d = os.path.join(TRAIN, study, ser)
    files = [e.path for e in os.scandir(d) if e.is_file() and e.name.lower().endswith(".dcm")]
    dss = [pydicom.dcmread(f) for f in files]
    dss.sort(key=slice_position)
    n = len(dss)
    idx = np.unique(np.linspace(0, n - 1, min(n, MAX_SLICES)).round().astype(int))
    imgs = []
    for i in idx:
        a = dss[i].pixel_array.astype(np.float32)
        a = a * float(getattr(dss[i], "RescaleSlope", 1)) + float(getattr(dss[i], "RescaleIntercept", 0))
        imgs.append(torch.from_numpy(a)[None, None])
    # slices of one series can differ in size: resize each to IMG first
    vol = torch.cat([F.interpolate(t, size=(IMG, IMG), mode="bilinear", align_corners=False) for t in imgs])[:, 0]
    lo, hi = np.percentile(vol.numpy(), [0.5, 99.5])
    vol = ((vol - lo) / max(hi - lo, 1e-6)).clamp(0, 1)
    z_centers = (idx + 0.5) / n  # normalized z of each kept slice, same convention as the box files
    return vol, z_centers

def read_study(study):
    t0 = time.time()
    out = {}
    for ser in series.loc[series.StudyInstanceUID == study, "SeriesInstanceUID"]:
        try:
            out[ser] = read_series(study, ser)
        except Exception as e:
            out[ser] = e
    return study, out, time.time() - t0

# ---------- pooling ----------
def pool(fmap, z_centers, box):
    """fmap: [S, C, h, w]; box normalized (x0, y0, z0, x1, y1, z1). Mean over cells inside the box."""
    S, C, h, w = fmap.shape
    cx = (torch.arange(w, device=fmap.device) + 0.5) / w
    cy = (torch.arange(h, device=fmap.device) + 0.5) / h
    cz = torch.tensor(z_centers, device=fmap.device, dtype=torch.float32)
    def sel(c, a, b):
        m = (c >= a) & (c <= b)
        if not m.any():  # box thinner than one cell: take the nearest cell
            m = torch.zeros_like(m); m[torch.argmin((c - (a + b) / 2).abs())] = True
        return m
    mz, my, mx = sel(cz, box[2], box[5]), sel(cy, box[1], box[4]), sel(cx, box[0], box[3])
    return fmap[mz][:, :, my][:, :, :, mx].float().mean(dim=(0, 2, 3))

@torch.no_grad()
def extract(vol, z_centers, box_df):
    x = vol.to(dev).half()[:, None].repeat(1, 3, 1, 1)
    x = (x - MEAN) / STD
    fmap = model.forward_features(x)            # [S, 768, 7, 7]
    feats = torch.zeros(13, fmap.shape[1], device=dev)
    for k, f in enumerate(FINDINGS):
        if box_df is not None and f in box_df.index:
            feats[k] = pool(fmap, z_centers, box_df.loc[f].values.astype(float))
    feats[12] = fmap.float().mean(dim=(0, 2, 3))
    return feats.half().cpu().numpy()

# ---------- run ----------
studies = series["StudyInstanceUID"].unique()
if N_STUDIES:
    studies = studies[:N_STUDIES]
done = {os.path.basename(p)[:-4] for p in glob.glob(f"{OUT}/*.npz")}
studies = [s for s in studies if s not in done]

t_start, t_read, t_gpu, n_slices, errors = time.time(), 0.0, 0.0, 0, []
with ThreadPoolExecutor(4) as ex:                 # read the next studies while the GPU works
    for i, (study, data, dt_read) in enumerate(ex.map(read_study, studies)):
        t_read += dt_read
        t0 = time.time()
        save = {}
        for ser, v in data.items():
            if isinstance(v, Exception):
                errors.append((study, ser, repr(v)[:120])); continue
            vol, zc = v
            save[ser] = extract(vol, zc, box_lookup.get(ser))
            n_slices += len(zc)
        torch.cuda.synchronize()
        t_gpu += time.time() - t0
        np.savez(f"{OUT}/{study}.npz", **save)
        if (i + 1) % 10 == 0 or i + 1 == len(studies):
            wall = time.time() - t_start
            print(f"{i + 1}/{len(studies)} studies | wall {wall:.0f} s | {wall / (i + 1):.2f} s/study | "
                  f"{n_slices / wall:.0f} slices/s | read (summed over threads) {t_read:.0f} s | gpu {t_gpu:.0f} s",
                  flush=True)

wall = time.time() - t_start
per_study = wall / max(len(studies), 1)
print(f"\nerrors: {len(errors)}")
for e in errors[:5]:
    print("  ", e)
print(f"\n== timing: {per_study:.2f} s/study  ->  all 4,407 training studies ~ {per_study * 4407 / 3600:.1f} h")
print(f"GPU share of wall time: {t_gpu / wall:.0%}  (low = DICOM reading is the bottleneck)")
f0 = np.load(glob.glob(f"{OUT}/*.npz")[0])
print("example file:", len(f0.files), "series, each", f0[f0.files[0]].shape, f0[f0.files[0]].dtype)

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

10/4407 studies | wall 14 s | 1.44 s/study | 113 slices/s | read (summed over threads) 37 s | gpu 13 s
20/4407 studies | wall 22 s | 1.09 s/study | 143 slices/s | read (summed over threads) 67 s | gpu 16 s
30/4407 studies | wall 31 s | 1.02 s/study | 151 slices/s | read (summed over threads) 103 s | gpu 20 s
40/4407 studies | wall 40 s | 0.99 s/study | 158 slices/s | read (summed over threads) 140 s | gpu 24 s
50/4407 studies | wall 48 s | 0.95 s/study | 165 slices/s | read (summed over threads) 173 s | gpu 28 s
60/4407 studies | wall 59 s | 0.98 s/study | 161 slices/s | read (summed over threads) 219 s | gpu 31 s
70/4407 studies | wall 73 s | 1.04 s/study | 155 slices/s | read (summed over threads) 264 s | gpu 36 s
80/4407 studies | wall 84 s | 1.05 s/study | 150 slices/s | read (summed over threads) 302 s | gpu 39 s
90/4407 studies | wall 88 s | 0.97 s/study | 162 slices/s | read (summed over threads) 339 s | gpu 43 s
100/4407 studies | wall 97 s | 0.97 s/study | 161 slices/s | read 